# Data Acquisition for Electricity Price Forecasting

This notebook handles downloading and caching all required data for both DE-LU and ES zones.

## Data Sources
1. **Historical DAA Prices**: energy-charts.info
2. **Generation Data**: ENTSO-E Transparency Platform
3. **Weather Data**: Open-Meteo API / ERA5
4. **Fuel Prices**: ICE/EEX market data

## Time Range
- Training data: 2020-01-01 to 2026-05-05 (current date)
- Evaluation window: 2026-05-08 18:00 to 2026-05-09 23:00

In [1]:
# Install dependencies if needed
%pip install -q pandas numpy requests tqdm

Note: you may need to restart the kernel to use updated packages.


In [16]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime, timedelta
from tqdm import tqdm

import src.data.loaders

print("Imports successful!")
print(f"Current date: {datetime.now()}")

Imports successful!
Current date: 2026-05-06 09:42:33.053426


In [3]:
import importlib
importlib.reload(src.data.loaders)

<module 'src.data.loaders' from 'c:\\Users\\Nutzer\\Documents\\Studium\\Bachelor Wirtschaftsinformatik\\Bewerbungen\\Auslandssemester\\Kurse\\Hackathon\\frigg_hack\\frigg_forecasting\\notebooks\\..\\src\\data\\loaders.py'>

In [4]:
from src.data.loaders import EnergyChartsLoader, WeatherDataLoader

## 1. Download Historical DAA Prices

In [5]:
# Initialize loader
energy_loader = EnergyChartsLoader(cache_dir=Path('../data/raw'))

# Define date range
start_date = '2026-01-01'
end_date = '2026-05-03'

zones = ['DE-LU', 'ES']

# Download prices for both zones
prices_data = {}

for zone in zones:
    print(f"\n{'='*60}")
    print(f"Downloading DAA prices for {zone}")
    print(f"{'='*60}")
    
    df = energy_loader.load_day_ahead_prices(
        zone=zone,
        start_date=start_date,
        end_date=end_date,
        use_cache=False  # Set to True after first run
    )
    
    prices_data[zone] = df
    
    print(f"\nLoaded {len(df):,} hourly records")
    print(f"Date range: {df['timestamp'].min()} to {df['timestamp'].max()}")
    print(f"\nPrice statistics for {zone}:")
    print(df['price_eur_mwh'].describe())
    print(f"\nNegative prices: {(df['price_eur_mwh'] < 0).sum()} hours ({(df['price_eur_mwh'] < 0).mean()*100:.2f}%)")


Cached data to ..\data\raw\de-lu\daa_prices_20260101_20260503.csv

Loaded 11,713 hourly records
Date range: 2026-01-01 00:00:00+00:00 to 2026-05-03 00:00:00+00:00

Price statistics for DE-LU:
count    11713.000000
mean        95.168707
std         57.162302
min       -499.990000
25%         81.640000
50%        101.830000
75%        123.660000
max        429.360000
Name: price_eur_mwh, dtype: float64

Negative prices: 723 hours (6.17%)

Cached data to ..\data\raw\es\daa_prices_20260101_20260503.csv

Loaded 11,713 hourly records
Date range: 2026-01-01 00:00:00+00:00 to 2026-05-03 00:00:00+00:00

Price statistics for ES:
count    11713.000000
mean        43.868733
std         45.010443
min        -10.000000
25%          0.980000
50%         31.920000
75%         82.200000
max        250.000000
Name: price_eur_mwh, dtype: float64

Negative prices: 1612 hours (13.76%)


## 2. Download Generation Data

In [6]:
# Download generation data for both zones
generation_data = {}

for zone in zones:
    print(f"\n{'='*60}")
    print(f"Downloading generation data for {zone}")
    print(f"{'='*60}")
    
    df = energy_loader.load_generation_data(
        zone=zone,
        start_date=start_date,
        end_date=end_date,
        use_cache=False
    )
    
    generation_data[zone] = df
    
    print(f"\nLoaded {len(df):,} hourly records")
    print(f"\nGeneration mix statistics for {zone}:")
    print(df.describe())



Loaded 11,713 hourly records

Generation mix statistics for DE-LU:
       wind_onshore  wind_offshore         solar  hydro_run-of-river  \
count  11713.000000   11713.000000  11713.000000        11713.000000   
mean   14073.724221    3984.420405   7525.308666         1480.467711   
std     9661.333562    2500.783790  12720.261241          259.998234   
min      213.300000       0.000000      0.000000         1059.100000   
25%     6241.800000    1731.000000      0.000000         1283.100000   
50%    12523.400000    3951.400000     30.600000         1419.100000   
75%    19050.300000    6289.500000   9970.300000         1639.100000   
max    46660.800000    8480.700000  54198.900000         2396.300000   

         fossil_gas  fossil_hard_coal    fossil_oil  fossil_coal-derived_gas  \
count  11713.000000      11713.000000  11713.000000             11713.000000   
mean    9027.364510       4429.303253    292.626483               525.348613   
std     4709.433769       2074.450223    1

c:\Users\Nutzer\Documents\Studium\Bachelor Wirtschaftsinformatik\Bewerbungen\Auslandssemester\Kurse\Hackathon\frigg_hack\.venv\Lib\site-packages\pandas\core\nanops.py:1027: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)
c:\Users\Nutzer\Documents\Studium\Bachelor Wirtschaftsinformatik\Bewerbungen\Auslandssemester\Kurse\Hackathon\frigg_hack\.venv\Lib\site-packages\pandas\core\nanops.py:1027: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


## 3. Download Weather Data

In [7]:
# Initialize weather loader
weather_loader = WeatherDataLoader(cache_dir=Path('../data/external'))

# Download weather data for both zones
weather_data = {}

for zone in zones:
    print(f"\n{'='*60}")
    print(f"Downloading weather data for {zone}")
    print(f"{'='*60}")
    
    df = weather_loader.load_weather_data(
        zone=zone,
        start_date=start_date,
        end_date=end_date,
        use_cache=False
    )
    
    weather_data[zone] = df
    
    print(f"\nLoaded {len(df):,} hourly records")
    print(f"\nWeather statistics for {zone}:")
    print(df.describe())


  Fetching historical weather: 2026-01-01 to 2026-05-03

Loaded 2,929 hourly records

Weather statistics for DE-LU:
       temperature_2m_c  wind_speed_10m_ms  wind_speed_100m_ms  \
count       2929.000000        2929.000000         2929.000000   
mean           3.672755           9.562376           18.186446   
std            6.028579           6.647382           10.237489   
min          -13.800000           0.000000            0.300000   
25%           -0.900000           4.400000            9.800000   
50%            3.200000           8.000000           17.500000   
75%            8.000000          13.900000           24.600000   
max           24.500000          32.500000           48.800000   

       wind_direction_10m_deg  solar_irradiance_wm2  cloud_cover_pct  \
count             2929.000000           2929.000000      2929.000000   
mean               185.684534            115.673267        66.427791   
std                 94.238221            190.138474        41.929981   


## 4. Data Quality Checks

In [ ]:
print("\n" + "="*60)
print("DATA QUALITY SUMMARY")
print("="*60)

for zone in zones:
    print(f"\n{zone}:")
    print("-" * 40)
    
    # Check for missing values
    print(f"Prices - Missing values: {prices_data[zone].isnull().sum().sum()}")
    print(f"Generation - Missing values: {generation_data[zone].isnull().sum().sum()}")
    print(f"Weather - Missing values: {weather_data[zone].isnull().sum().sum()}")
    
    # Check timestamp alignment
    print(f"\nTimestamp alignment:")
    print(f"  Prices: {len(prices_data[zone])} records")
    print(f"  Generation: {len(generation_data[zone])} records")
    print(f"  Weather: {len(weather_data[zone])} records")
    
    # Check for duplicates
    print(f"\nDuplicate timestamps:")
    print(f"  Prices: {prices_data[zone]['timestamp'].duplicated().sum()}")
    print(f"  Generation: {generation_data[zone]['timestamp'].duplicated().sum()}")
    print(f"  Weather: {weather_data[zone]['timestamp'].duplicated().sum()}")

## 5. Create Merged Dataset

In [ ]:
# Merge all data sources for each zone
merged_data = {}

for zone in zones:
    print(f"\nMerging data for {zone}...")
    
    # Start with prices
    df = prices_data[zone].copy()
    
    # Merge generation data
    df = df.merge(
        generation_data[zone],
        on='timestamp',
        how='left'
    )
    
    # Merge weather data
    df = df.merge(
        weather_data[zone],
        on='timestamp',
        how='left'
    )
    
    # Sort by timestamp
    df = df.sort_values('timestamp').reset_index(drop=True)
    
    merged_data[zone] = df
    
    # Save merged dataset
    output_path = Path(f'../data/processed/{zone.lower()}_merged.csv')
    output_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(output_path, index=False)
    
    print(f"Saved merged dataset to {output_path}")
    print(f"Shape: {df.shape}")
    print(f"Columns: {list(df.columns)}")

## 6. Quick Visualization

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')

# Plot price comparison
fig, axes = plt.subplots(2, 1, figsize=(15, 8))

for idx, zone in enumerate(zones):
    df = merged_data[zone]
    
    # Sample last 30 days for visualization
    df_recent = df.tail(30 * 24)
    
    axes[idx].plot(df_recent['timestamp'], df_recent['price_eur_mwh'], linewidth=0.8)
    axes[idx].set_title(f'{zone} - Day Ahead Prices (Last 30 Days)', fontsize=12, fontweight='bold')
    axes[idx].set_ylabel('Price (EUR/MWh)')
    axes[idx].grid(True, alpha=0.3)
    axes[idx].axhline(y=0, color='r', linestyle='--', alpha=0.5, label='Zero price')
    axes[idx].legend()

plt.tight_layout()
plt.savefig('../outputs/price_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nVisualization saved to outputs/price_comparison.png")

## Summary

Data acquisition complete! We now have:

1. ✅ Historical DAA prices (2020-2026) for DE-LU and ES
2. ✅ Generation data by source
3. ✅ Weather data (temperature, wind, solar)
4. ✅ Merged datasets ready for feature engineering

Next steps:
- Notebook 02: Exploratory Data Analysis
- Notebook 03: Feature Engineering
- Notebook 04: Model Development